In [59]:
import time
from tqdm import tqdm
import pandas as pd
import urllib.parse

from dynamic import SeleniumScraperClient
from static import BeautifulSoupScraperClient

In [88]:
# workaround via specifying an invalid value first
# %config Application.log_level='WORKAROUND'
# => fails, necessary on Fedora 27, ipython3 6.2.1
%config Application.log_level='INFO'
import logging
logging.getLogger().setLevel(40)
log = logging.getLogger()

In [89]:
log.info('Test info')
log.warning('Test warning')
log.debug('Test debug')
log.critical('Test critical')
log.error('Test error')

CRITICAL:root:Test critical
ERROR:root:Test error


In [2]:
import requests
from bs4 import BeautifulSoup
import re

# This is the page listing donors and their country filters.
url = "https://fts.unocha.org/global-funding/donors/2025"

# 1) Send a GET request to fetch the page
response = requests.get(url)
if not response.ok:
    raise Exception(f"Failed to retrieve the page, status code: {response.status_code}")


In [48]:
# 2) Parse the HTML with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# Initialize dictionaries for each facet type.
countries = {}
donors = {}
sectors = {}

# Pre-compile regex patterns for efficiency.
country_pattern = re.compile(r"destinationlocationidname-(\d+)", re.IGNORECASE)
donor_pattern = re.compile(r"sourceorganizationidname-(\d+)", re.IGNORECASE)
sector_pattern = re.compile(r"destinationglobalclusteridname-(\d+)", re.IGNORECASE)

# Loop through each facet item (rendered as <li class="facet-item">)
for item in soup.select("li.facet-item"):
    # Get the anchor containing the data attribute.
    a_tag = item.find("a", attrs={"data-drupal-facet-item-id": True})
    if not a_tag:
        continue

    # Extract the filter name from the <span> text.
    label = a_tag.find("span", class_="facet-item__value")
    if not label:
        continue
    facet_text = label.get_text(strip=True)
    
    # Skip unwanted options (like "None" or "!" if applicable).
    if facet_text.lower() == "none" or facet_text == "!":
        continue

    # Retrieve the data facet id.
    data_id = a_tag.get("data-drupal-facet-item-id", "")

    # Check for a country filter.
    country_match = country_pattern.search(data_id)
    if country_match:
        countries[facet_text] = int(country_match.group(1))
        continue

    # Check for a donor filter.
    donor_match = donor_pattern.search(data_id)
    if donor_match:
        donors[facet_text] = int(donor_match.group(1))
        continue

    # Check for a sector filter.
    sector_match = sector_pattern.search(data_id)
    if sector_match:
        sectors[facet_text] = int(sector_match.group(1))
        continue

In [49]:
# Display the extracted dictionaries.
print("Extracted Countries:")
for key, value in countries.items():
    print(f"  {key}: {value}")

Extracted Countries:
  Afghanistan: 1
  Albania: 3
  Algeria: 4
  Angola: 7
  Argentina: 11
  Armenia: 12
  Aruba (Netherlands): 13
  Australia: 14
  Bangladesh: 19
  Benin: 24
  Bolivia, Plurinational State of: 27
  Bosnia and Herzegovina: 28
  Botswana: 30
  Brazil: 32
  Brunei Darussalam: 34
  Bulgaria: 35
  Burkina Faso: 36
  Burundi: 37
  Cambodia: 38
  Cameroon: 39
  Canada: 40
  Central African Republic: 43
  Chad: 44
  Chile: 45
  Colombia: 49
  Comoros: 50
  Congo: 51
  Cook Islands: 53
  Costa Rica: 54
  Côte d'Ivoire: 55
  Croatia: 56
  Cuba: 57
  Curaçao (Netherlands): 58
  Czech Republic: 60
  Democratic Republic of the Congo: 52
  Djibouti: 62
  Dominican Republic: 64
  Ecuador: 65
  Egypt: 66
  El Salvador: 67
  Eritrea: 69
  Estonia: 70
  Eswatini: 215
  Ethiopia: 71
  Fiji: 74
  France: 76
  Gabon: 80
  Gambia: 81
  Georgia: 82
  Ghana: 84
  Global: 251
  Grenada: 88
  Guatemala: 91
  Guinea: 93
  Guyana: 95
  Haiti: 96
  Honduras: 99
  Hungary: 101
  India: 103
  Indo

In [50]:
print("\nExtracted Donors:")
for key, value in donors.items():
    print(f"  {key}: {value}")


Extracted Donors:
  Access to Health Fund: 12393
  Action Contre la Faim - Action Against Hunger International: 4536
  Action Contre la Faim - France: 8501
  Adventist Development and Relief Agency: 4695
  African Development Bank: 3787
  Agence Française de Développement: 9946
  Aid Fund for Northern Syria: 13209
  Aktion Deutschland Hilft: 2398
  Al-Salam Association for humanitarian and charitable works: 14712
  Algeria, Government of: 4542
  American Red Cross: 105
  Armenia, Government of: 2969
  Australia, Government of: 4391
  Austria, Government of: 4546
  Azerbaijan, Government of: 2783
  Bangladesh, Government of: 2981
  Belgium, Government of: 2927
  Bhutan, Government of: 1484
  Bill and Melinda Gates Foundation: 4326
  Brazil, Government of: 4838
  Bulgaria, Government of: 5281
  Cambodia, Government of: 3349
  Canada, Government of: 2928
  Canadian Food Grains Bank: 1892
  Canadian Red Cross Society: 2907
  Caritas Germany (DCV): 3162
  Centers for Disease Control and Pr

In [51]:
print("\nExtracted Sectors:")
for key, value in sectors.items():
    print(f"  {key}: {value}")


Extracted Sectors:
  Agriculture: 26512
  Camp Coordination / Management: 1
  Coordination and support services: 26480
  Early Recovery: 2
  Education: 3
  Emergency Shelter and NFI: 4
  Emergency Telecommunications: 5
  Food Security: 6
  Health: 7
  Logistics: 8
  Multi-sector: 26479
  Multipurpose Cash: 16
  Nutrition: 9
  Other: 26481
  Protection: 10
  Protection - Child Protection: 12
  Protection - Gender-Based Violence: 13
  Protection - Housing, Land and Property: 14
  Protection - Human Trafficking & Smuggling: 26546
  Protection - Mine Action: 15
  Water Sanitation Hygiene: 11


In [12]:
len(countries), len(sectors)

(151, 21)

In [90]:
import urllib.parse
import pandas as pd
import time
import os
import pickle

def generate_unocha_url(country_name: str = None, country_id: int = None, 
                         sector_name: str = None, sector_id: int = None,
                         base_url: str = "https://fts.unocha.org/global-funding/donors/2025") -> str:
    """
    Generate a UNOCHA URL using provided country and sector filters.
    """
    params = {}
    index = 0
    if country_name and country_id:
        params[f"f[{index}]"] = f"destinationLocationIdName:{country_id}:{country_name}"
        index += 1
    if sector_name and sector_id:
        params[f"f[{index}]"] = f"destinationGlobalClusterIdName:{sector_id}:{sector_name}"
        index += 1
    query_string = urllib.parse.urlencode(params)
    return f"{base_url}?{query_string}"


def loop_through_unocha_urls(client, 
                              countries: dict, 
                              sectors: dict,
                              table_index: int = 0, 
                              paginated: bool = False,
                              next_button_xpath: str = None, 
                              sleep_time: int = 2,
                              max_retries: int = 3,
                              retry_delay: int = 3,
                              checkpoint_file: str = "checkpoint.pkl") -> tuple:
    """
    Loop over all (country, sector) combinations, scraping each URL.
    After processing each country, the country-level DataFrame is saved to a checkpoint file.
    If the checkpoint file exists, processing resumes only with missing countries.
    
    :return: A tuple (country_dfs, combined_df)
    """
    # Load checkpoint if available
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "rb") as f:
            country_dfs = pickle.load(f)
        log.info(f"Loaded checkpoint with {len(country_dfs)} countries processed already.")
    else:
        country_dfs = {}

    all_dfs = []
    
    # Loop over each country; skip countries already processed
    for country, country_id in (pbar := tqdm(countries.items(), desc="Countries")):
        pbar.set_postfix_str(country)
        if country in country_dfs:
            log.info(f"Skipping {country} (already processed).")
            all_dfs.append(country_dfs[country])
            continue
        
        sector_dfs = []
        for sector, sector_id in sectors.items():
            url = generate_unocha_url(country, country_id, sector, sector_id)
            log.debug("Processing URL:", url)
            attempt = 0
            success = False
            df = None

            while attempt < max_retries and not success:
                try:
                    if isinstance(client, SeleniumScraperClient):
                        df = client.scrape_table_by_index(
                            url, table_index=table_index, paginated=paginated,
                            next_button_xpath=next_button_xpath, sleep_time=sleep_time
                        )
                    elif isinstance(client, BeautifulSoupScraperClient):
                        df = client.scrape_table_by_index(
                            url, table_index=table_index, paginated=paginated,
                            next_button_selector=next_button_xpath, sleep_time=sleep_time
                        )
                    else:
                        raise TypeError("Unsupported client type provided.")
                    success = True
                except IndexError:
                    log.warning(f"IndexError: This may indicate no data available for {country} - {sector}.")
                    break  # Exit the retry loop if no data is available.
                except Exception as e:
                    attempt += 1
                    log.debug(f"Attempt {attempt}/{max_retries} failed for {country} - {sector}: {e}")
                    time.sleep(retry_delay)
            
            if success and df is not None and not df.empty:
                df['Country'] = country
                df['Sector'] = sector
                df['URL'] = url
                sector_dfs.append(df)
                all_dfs.append(df)
            else:
                log.debug(f"Skipping {country} - {sector} after {attempt} failed attempts.")
            time.sleep(1)  # Optional delay between requests
        
        # Merge data from all sectors for the current country into one DataFrame.
        if sector_dfs:
            country_df = pd.concat(sector_dfs, ignore_index=True)
        else:
            country_df = pd.DataFrame()
        country_dfs[country] = country_df
        
        # Save checkpoint to file after processing each country.
        with open(checkpoint_file, "wb") as f:
            pickle.dump(country_dfs, f)
        log.info(f"Checkpoint saved; processed country: {country}")

    # Combine data from all countries.
    combined_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
    return country_dfs, combined_df

In [91]:
# tmp = pickle.load(open("checkpoint.pkl", "rb"))
# tmp_countries = list(tmp.keys())
# index = tmp_countries.index("Gabon")

# # Remove all entries after and including "Gabon"
# tmp_countries = tmp_countries[:index]
# tmp = {key: value for key, value in tmp.items() if key in tmp_countries}

In [92]:
# tmp.keys()

In [93]:
# pickle.dump(tmp, open("checkpoint.pkl", "wb"))

In [ ]:
# Replace with your scraping client instance:
client = BeautifulSoupScraperClient()  

# # Test cases for countries and sectors.:
# countries = {"Afghanistan": 1, "Cambodia": 38, "Yemen": 248}
# sectors = {"Agriculture": 26512, "Health": 26518, "Education": 26515}

# Run with checkpointing enabled.
country_dfs, combined_df = loop_through_unocha_urls(
    client,
    countries,
    sectors,
    table_index=1,            # Adjust according to the target table index.
    paginated=False,          # Set True if pagination is required.
    max_retries=2,
    retry_delay=3,
    checkpoint_file="checkpoint.pkl"
)

Countries:  69%|██████▉   | 104/151 [04:31<06:24,  8.18s/it, Philippines]                  

In [ ]:
# checkpoint_file = "checkpoint.pkl"
# with open(checkpoint_file, "rb") as f:
#     checkpoint = pickle.load(f)

# # For each country in the checkpoint, update the DataFrame by adding the "URL" column.
# for country, df in checkpoint.items():
#     if df.empty:
#         continue
#     # Define a helper function that generates the URL based on row contents.
#     def get_url(row):
#         # We assume each row has a "Country" and "Sector" column.
#         c_name = row.get("Country", country)  # Fall back to the dict key if missing.
#         s_name = row.get("Sector")
#         # Look up the numeric IDs from your dictionaries.
#         try:
#             c_id = countries[c_name]
#             s_id = sectors[s_name]
#         except KeyError:
#             return None
#         return generate_unocha_url(c_name, c_id, s_name, s_id)
    
#     # Apply the helper function to add/update the 'URL' column.
#     df['URL'] = df.apply(get_url, axis=1)
#     checkpoint[country] = df

# # Save the updated checkpoint back to disk.
# with open(checkpoint_file, "wb") as f:
#     pickle.dump(checkpoint, f)

# print("Checkpoint updated: URL column added for each entry.")

Checkpoint updated: URL column added for each entry.


In [ ]:
# checkpoint['Gabon']['URL'].iloc[0]

'https://fts.unocha.org/global-funding/donors/2025?f%5B0%5D=destinationLocationIdName%3A80%3AGabon&f%5B1%5D=destinationGlobalClusterIdName%3A26481%3AOther'

In [18]:
combined_df.to_csv("combined_data.csv", index=False)

In [17]:
# Optionally, inspect the combined DataFrame:
print("Combined DataFrame (first 5 rows):")
combined_df.head()

Combined DataFrame (first 5 rows):


,Source org.Sort descending,Funding US$,Pledges US$,Country,Sector
0,"Italy, Government of","793,159",0,Afghanistan,Agriculture
1,"Switzerland, Government of","2,155,689",0,Afghanistan,Agriculture
2,"United Kingdom, Government of","63,847",0,Afghanistan,Camp Coordination / Management
3,European Commission's Humanitarian Aid and Civ...,"557,804",0,Afghanistan,Coordination and support services
4,"Germany, Government of","274,262",0,Afghanistan,Coordination and support services


In [22]:
# Or inspect a specific country’s data:
for country, df in country_dfs.items():
    print(f"\nData for {country} (first 5 rows):")
    print(df.head())


Data for Afghanistan (first 5 rows):
   Source org.Sort descending Funding US$ Pledges US$      Country  \
0        Italy, Government of     793,159           0  Afghanistan   
1  Switzerland, Government of   2,155,689           0  Afghanistan   

        Sector  
0  Agriculture  
1  Agriculture  

Data for Ethiopia (first 5 rows):
                Source org.Sort descending Funding US$ Pledges US$   Country  \
0                 Australia, Government of   1,191,614           0  Ethiopia   
1  United States of America, Government of   1,130,000           0  Ethiopia   

        Sector  
0  Agriculture  
1  Agriculture  

Data for Ukraine (first 5 rows):
                Source org.Sort descending Funding US$ Pledges US$  Country  \
0                 Australia, Government of   1,191,614           0  Ukraine   
1  United States of America, Government of   2,559,202           0  Ukraine   

        Sector  
0  Agriculture  
1  Agriculture  
